In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import scienceplots

from FastBEMT import (
    BEMT,
    Environment,
    Propeller,
    Simulation,
    load_propeller_geometries,
)
from FastBEMT.Utils import BladeStressCalculator, Plotter

plt.style.use(["science", "no-latex"])

geometry_directory = Path(
    r"C:\Users\PusinoDavide\LFProp\Datasets\Optimization Results\NSGAII\Parametric\NewGAN\run_002"
)
propeller_geometries = load_propeller_geometries(geometry_directory)
print(f"Loaded {len(propeller_geometries)} propellers")

rpm = 7000.0
material_density = 2700.0  # kg/m^3
environment = Environment()
simulation = Simulation(
    revolutions=5,
    timesteps_per_revolution=100,
    device="cuda",
)


In [ ]:
figure, (chord_axis, twist_axis) = plt.subplots(1, 2, figsize=(12, 4))

for propeller_name, geometry in propeller_geometries:
    radius = geometry["r"]
    chord_axis.plot(
        radius,
        geometry["chord"],
        marker="o",
        markersize=4,
        label=propeller_name,
        alpha=0.7,
    )
    twist_axis.plot(
        radius,
        geometry["twist"],
        marker="s",
        markersize=4,
        label=propeller_name,
        alpha=0.7,
    )

for axis, ylabel, title in (
    (chord_axis, "Chord [m]", "Chord distribution"),
    (twist_axis, "Twist [deg]", "Twist distribution"),
):
    axis.set_xlabel("Radial position [m]")
    axis.set_ylabel(ylabel)
    axis.set_title(title)
    axis.grid(True, linestyle=":")
    axis.legend(frameon=True, edgecolor="none")
    axis.spines[["top", "right"]].set_visible(False)
    axis.tick_params(top=False, right=False, which="both")

figure.tight_layout()
plt.show()


In [ ]:
output_directory = Path("../Figures/Blade Renderings")
output_directory.mkdir(parents=True, exist_ok=True)

for propeller_name, geometry in propeller_geometries:
    print(f"Processing {propeller_name}...")
    propeller = Propeller(geometry, environment, simulation)
    bemt = BEMT(propeller, rpm=rpm, v_inf=0.0)

    stress_calculator = BladeStressCalculator(propeller)
    centrifugal_stress, bending_stress = stress_calculator.compute_stress(
        material_density,
        bemt,
    )

    Plotter(propeller).plot_stress_distribution(
        centrifugal_stress,
        bending_stress,
        figsize=(4.5, 3.0),
        cmap="viridis",
        save_path=output_directory / f"{propeller_name}_stress.png",
    )

print("All propellers processed.")
